# WS-9 — Animação Científica da Simulação (Groveman-clock humanizado)
**Dois vídeos, CPU basta (sem GPU):**
- **A — SEM tratamento:** semente → eclipse → replicação exponencial → frente avança → doença estabelecida
- **B — COM tratamento V127ΔGPI:** depósito secretor → competição de substrato → gradiente → casca de contenção → quase-extinção

Pausas com legenda breve em cada etapa (frames congelados). Relógio: 1 unidade sim = 144 dias reais (âncoras Groveman 2019).
Outputs: `ws9_anim_A_sem.mp4`, `ws9_anim_B_com.mp4` + GIFs + frame-chave PNG. Downloads automáticos.

In [ ]:
#@title C0 — motor + captura de frames (~5s) {display-mode:"form"}
import numpy as np, json, math, os, time
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
os.makedirs('/content/out', exist_ok=True); T0=time.time()

DAYS_PER_UNIT = 144.0  # relógio humano (WS-9 v4, âncoras Groveman 2019)

def simulate_frames(div=96, s=10, t_lim=5.0, dt=5e-4, kcap=0.0, ell_mm=3.6,
             Kt=(10.0,5.0), Kr=(50.0,10.0), Kc=(10.0,50.0), D0=1000.0, L=1.0,
             uprd=5.0, uprt=10.0, uprr=6.0, tpr=10.0, C50=50.0, seed_mass=130.0,
             nframes=90, tag='', progress=True):
    """Simula e captura nframes snapshots (carga total + campo V127)."""
    t_start=time.time()
    px_per_mm=div/4.0
    K_templ,K_auto,K_nucl,K_frag,K_decond,K_cond=Kt[1],Kt[0],Kr[0],Kr[1],Kc[0],Kc[1]
    K1=4*D0/(np.arange(1,s+2)*L**2)
    X,Y=np.meshgrid(np.arange(1,div+1),np.arange(1,div+1))
    m_lin=round(uprr+(div-2*uprr)/2); step=max(1,round((div-2*uprr-1)/2))
    neur=[(m_lin+step*i,m_lin+step*j) for i in(-1,0,1) for j in(-1,0,1)]
    def disk(cx,cy,r): return ((X-cx)**2+(Y-cy)**2)<r**2
    tpl=[disk(*p,tpr) for p in neur]; upz=[disk(*p,uprr) for p in neur]
    c0=(div//2,div//2)
    P=np.zeros((div,div,s+1),dtype=float)
    sm=disk(c0[0],c0[1],3.0); P[sm,s]=seed_mass/sm.sum()
    rr=np.hypot(X-c0[0],Y-c0[1])/px_per_mm
    cV=np.exp(-rr/ell_mm) if kcap>0 else np.zeros((div,div))
    upr_t=np.zeros(9); upr_on=np.zeros(9,bool); tp=np.ones((div,div))
    steps=int(t_lim/dt); cap_every=max(1,steps//nframes)
    F_t=[]; F_load=[]; F_R=[]; F_C=[]; F_upr=[]
    marks={int(steps*f):f for f in (0.25,0.5,0.75)}
    for st in range(steps):
        if st%20==0:
            for n,(um,tm) in enumerate(zip(upz,tpl)):
                if P[um].sum()>uprd:
                    upr_t[n]+=dt*20
                    if upr_t[n]>=uprt: upr_on[n]=True
            tp.fill(1.0)
            for n,(um,tm) in enumerate(zip(upz,tpl)):
                if upr_on[n]: tp[tm]=0.0
        eff=tp
        C=P[:,:,s]; dP=np.zeros_like(P)
        freeS=(1.0/(1.0+kcap*cV))**2 if kcap>0 else np.ones((div,div))
        dP[:,:,s]+=dt*K_auto*eff*C*(C/(C+C50))*freeS
        for a in range(s-1):
            g=dt*K_templ*eff*P[:,:,a]*freeS
            dP[:,:,a]-=g; dP[:,:,a+1]+=g
        nuc=dt*K_nucl*C*freeS
        dP[:,:,0]+=nuc; dP[:,:,s]-=nuc
        frs=dt*K_frag*C[:,:,None]*P[:,:,1:s]
        dP[:,:,1:s]-=frs; dP[:,:,0:s-1]+=frs; dP[:,:,s]+=frs.sum(axis=2)
        dcs=dt*K_decond*P[:,:,1:s]
        dP[:,:,1:s]-=dcs; dP[:,:,0:s-1]+=dcs; dP[:,:,0]+=dcs.sum(axis=2)
        for a in range(s):
            for b in range(max(1,1-a),s-a):
                cr=dt*K_cond*P[:,:,a]*P[:,:,b]/(div*div)*10
                dP[:,:,a]-=cr; dP[:,:,b]-=cr; dP[:,:,a+b]+=2*cr
        lap=(np.roll(P,1,0)+np.roll(P,-1,0)+np.roll(P,1,1)+np.roll(P,-1,1)-4*P)
        dP+=dt*K1[None,None,:]/(div*div)*lap
        P=np.clip(P+dP,0,1e6)
        if st%cap_every==0:
            load=P.sum(axis=2)
            ys,xs=np.nonzero(load>1e-9)
            Rmax=float(np.max(np.hypot(xs-c0[0],ys-c0[1]))/px_per_mm) if len(xs) else 0.
            F_t.append(st*dt); F_load.append(load); F_R.append(Rmax); F_C.append(float(C.sum())); F_upr.append(float(upr_on.mean()))
        if progress and st in marks:
            el=time.time()-t_start
            print(f'  [{tag}] {int(marks[st]*100)}%  elapsed {el:.0f}s  ETA {el/st*(steps-st):.0f}s', flush=True)
    return dict(t=np.array(F_t), load=np.array(F_load), R=np.array(F_R), Ctot=np.array(F_C),
                upr=np.array(F_upr), cV=cV, wall=time.time()-t_start)
print('motor de frames OK')

In [ ]:
#@title C1 — RODAR SIM A: SEM TRATAMENTO (~1-2 min, progresso) {display-mode:"form"}
print('SIM A — doença natural (sem V127)...')
A=simulate_frames(kcap=0.0, tag='A-natural')
print(f"A: {len(A['t'])} frames | R final {A['R'][-1]:.2f} mm | carga final {A['Ctot'][-1]:.2e}")
assert A['Ctot'][-1]>A['Ctot'][0]*1.5, "baseline não replicou"


In [ ]:
#@title C2 — RODAR SIM B: COM V127 (k=8, acima do limiar 0.333) (~1-2 min) {display-mode:"form"}
print('SIM B — terapia V127ΔGPI (κ=8)...')
B=simulate_frames(kcap=8.0, tag='B-terapia')
print(f"B: {len(B['t'])} frames | R final {B['R'][-1]:.2f} mm | carga final {B['Ctot'][-1]:.2e}")
assert B['R'][-1]<0.9*A['R'][-1], "capping nao conteve"


In [ ]:
#@title C3 — helpers de render (pausa+legenda) (~5s) {display-mode:"form"}
from matplotlib import animation
from IPython.display import HTML, display

def frame_ax(ax, load, cV, sim, i, kcap):
    ax.clear()
    im=ax.imshow(np.log1p(load), cmap='inferno', vmin=0, vmax=np.log1p(1e4), animated=True)
    if kcap>0:
        cs=ax.contour(np.arange(sim['cV'].shape[1]), np.arange(sim['cV'].shape[0]), sim['cV'],
                      levels=[0.5,0.8], colors=['#39c6ff','#bfe9ff'], linewidths=[1.2,0.7], linestyles='--')
    dias=sim['t'][i]*DAYS_PER_UNIT
    ax.set_title(f"dia {dias:4.0f}  ·  frente R={sim['R'][i]:.2f} mm  ·  carga {sim['Ctot'][i]:.1e}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    return im

def make_video(sim, kcap, events, fname, fps=6, hold=8):
    """events: lista (idx_frame, título, legenda) — congela 'hold' frames com texto."""
    idxs=set()
    for i,_,_ in events: idxs.update(range(i, min(i+hold, len(sim['t']))))
    seq=[(i, next((e for e in events if e[0]<=i<e[0]+hold), None)) for i in range(len(sim['t']))]
    fig,ax=plt.subplots(figsize=(5.2,5.8)); plt.subplots_adjust(bottom=0.24)
    ims=[]
    for i,(idx,ev) in enumerate(seq):
        im=frame_ax(ax, sim['load'][idx], sim['cV'], sim, idx, kcap)
        txt=''
        if ev:
            _,titulo,legenda=ev
            txt=f"{titulo}\n{legenda}"
            ax.set_xlabel(txt, fontsize=8.5, color='#0b4f8a', wrap=True)
        ims.append([im])
    ani=animation.ArtistAnimation(fig, ims, interval=1000//fps, blit=False)
    ani.save(fname, writer=animation.FFMpegWriter(fps=fps))
    plt.close(fig)
    print('salvo', fname)
    return fname

In [ ]:
#@title C4 — VÍDEO A: SEM TRATAMENTO (pausas explicadas) (~1 min render) {display-mode:"form"}
n=len(A['t'])
EV_A=[
 (0,  '① SEMENTE', 'Inóculo priônico no centro — início da infecção (equivalente ao seeding do organoide)'),
 (int(n*0.10), '② ECLIPSE', 'Fase silenciosa: inóculo é processado; carga baixa ~indetectável (25-35 dias no organoide real)'),
 (int(n*0.28), '③ REPLICAÇÃO EXPONENCIAL', 'Auto-catálise C→2C + fragmentação: pontas ativas dobram — a PG 2^n do protocolo v0, agora com saturação real'),
 (int(n*0.50), '④ FRENTE DE CONVERSÃO', 'Onda priônica avança pelo parênquima: conversão templada + difusão intersticial (~0,1-1 mm/semana tecidual)'),
 (int(n*0.78), '⑤ UPR / RESISTÊNCIA TECIDUAL', 'Neurônios estressados desligam templating local — a doença contorna, não para'),
 (n-6,          '⑥ DOENÇA ESTABELECIDA', 'Substrato saturado: sem tratamento, o tecido todo é consumido — letalidade 100%'),
]
make_video(A, 0.0, EV_A, '/content/out/ws9_anim_A_sem.mp4')

In [ ]:
#@title C5 — VÍDEO B: COM V127 (pausas explicadas) (~1 min render) {display-mode:"form"}
n=len(B['t'])
EV_B=[
 (0,  '① SEMENTE + DEPÓSITO', 'Mesma infecção inicial — agora com depósito secretor de V127ΔGPI no centro (linhas azuis = campo do escudo, e^-r/ℓ)'),
 (int(n*0.10), '② CAMPO DO ESCUDO', 'A proteína V127 difunde do depósito: satura o interstício numa casca de alguns mm (WS-7: halo 4-6 mm)'),
 (int(n*0.30), '③ COMPETIÇÃO DE SUBSTRATO', 'PrP^C nativo disputado: onde V127 é abundante, a conversão fica sem matéria-prima (exclusão competitiva)'),
 (int(n*0.50), '④ GRADIENTE DE CONTENÇÃO', 'Inibição ~ 1/(1+κc)²: conversão suprimida perto do depósito, preservada longe — o gradiente proximal>distal que o G0 medirá'),
 (int(n*0.75), '⑤ CASCA DE CONTENÇÃO r*', 'A frente atinge a região do escudo e MORRE — θ<0,333: abaixo do limiar, a onda não se sustenta (bifurcação FKPP)'),
 (n-6,          '⑥ QUASE-EXTINÇÃO', 'Carga residual volta à semente: contenção robusta — a doença cercada, tecido vizinho preservado'),
]
make_video(B, 8.0, EV_B, '/content/out/ws9_anim_B_com.mp4')

In [ ]:
#@title C6 — GIFs + frame comparativo final + downloads (~40s) {display-mode:"form"}
# GIFs leves (sem som) para compartilhar
for tag,src in (('A','/content/out/ws9_anim_A_sem.mp4'),('B','/content/out/ws9_anim_B_com.mp4')):
    os.system(f"ffmpeg -y -loglevel error -i {src} -vf 'fps=4,scale=480:-1' /content/out/ws9_anim_{tag}.gif")
# frame lado-a-lado (dia final)
fig,axs=plt.subplots(1,2,figsize=(11,5))
for ax,sim,t in ((axs[0],A,'SEM tratamento'),(axs[1],B,'COM V127ΔGPI (κ=8)')):
    ax.imshow(np.log1p(sim['load'][-1]),cmap='inferno',vmin=0,vmax=np.log1p(1e4))
    ax.set_title(f"{t} — dia {sim['t'][-1]*DAYS_PER_UNIT:.0f}  R={sim['R'][-1]:.2f}mm  carga {sim['Ctot'][-1]:.1e}",fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig('/content/out/ws9_anim_final_compare.png',dpi=140); plt.show()
from google.colab import files
for f in ('ws9_anim_A_sem.mp4','ws9_anim_B_com.mp4','ws9_anim_A.gif','ws9_anim_B.gif','ws9_anim_final_compare.png'):
    fp=f'/content/out/{f}'
    if os.path.exists(fp): files.download(fp)
print('runtime total:', round(time.time()-T0,1),'s — CPU, sem GPU')